# OCSD – Ứng dụng minh họa (Gradio)

Vẽ phác thảo (hoặc tải ảnh), nhập mô tả từng đối tượng theo thứ tự trái → phải và mô tả nền; hệ thống tự phân tách đối tượng (M1) rồi sinh ảnh bằng OCSD-lite (không cần huấn luyện, nhanh) hoặc OCSD đầy đủ. Bảng kiểm tra cho biết số đối tượng phát hiện được.

In [ ]:
#@title 1. Cấu hình (chỉ sửa ô này)
REPO_URL   = "https://github.com/bobbibao/ocsd-thesis.git"  #@param {type:"string"}
BRANCH     = "main"                                             #@param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/KLTN_OCSD"                 #@param {type:"string"}
TIER       = "pilot"   #@param ["pilot", "paper", "full"]
BACKBONE   = "sd15"    #@param ["sd15", "sd21"]
MAX_MINUTES = 600      #@param {type:"number"}
HF_TOKEN   = ""        #@param {type:"string"}
# pilot: kiểm tra nhanh (~1-2 giờ GPU); paper: đủ số liệu cho khóa luận; full: cho bài báo (cần thêm GPU).
# Có thể bấm "Run all" nhiều lần: ảnh/độ đo đã có trên Drive sẽ được bỏ qua và chạy tiếp phần còn lại.

In [ ]:
#@title 2. Gắn Google Drive, kiểm tra GPU, lấy mã nguồn mới nhất
import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True, capture_output=True, text=True).stdout)
CODE = "/content/KLTN_OCSD"
if not REPO_URL:
    # chưa có repo: dùng bản mã nguồn đã tải lên Drive (thư mục DRIVE_ROOT/code)
    CODE = os.path.join(DRIVE_ROOT, "code")
elif os.path.exists(CODE):
    subprocess.run(f"git -C {CODE} fetch -q origin {BRANCH} && git -C {CODE} reset -q --hard origin/{BRANCH}", shell=True, check=True)
else:
    subprocess.run(f"git clone -q -b {BRANCH} {REPO_URL} {CODE}", shell=True, check=True)
print("Mã nguồn:", CODE, subprocess.run(f"git -C {CODE} log -1 --format='%h %s' 2>/dev/null", shell=True, capture_output=True, text=True).stdout)
sys.path.insert(0, CODE)
# drop modules imported by an earlier run in this kernel, so the code just pulled is what runs
for _m in [m for m in sys.modules if m == "ocsd" or m.startswith("ocsd.")]: del sys.modules[_m]
# HF cache on LOCAL disk: Drive cannot store the cache's symlinks (broken files -> "SafetensorError: header too large").
# Models re-download each new session (~2-4 min on Colab).
os.environ["HF_HOME"] = "/content/hf_cache"
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

In [ ]:
#@title 3. Install libraries (keeps Colab's PyTorch), start the Drive run log
LOGS = os.path.join(DRIVE_ROOT, "results", "logs"); os.makedirs(LOGS, exist_ok=True)
# Colab ships torchao 0.10, which peft rejects when adding LoRA ("incompatible version of torchao"); OCSD does not use it
subprocess.run("pip uninstall -y -q torchao", shell=True)
r = subprocess.run(f"pip install -q -r {CODE}/requirements-colab.txt", shell=True, capture_output=True, text=True)
open(os.path.join(LOGS, "pip_install.log"), "w").write(r.stdout + "\n" + r.stderr)
print(r.stdout[-2000:], r.stderr[-3000:])
if r.returncode != 0:
    raise RuntimeError(f"pip install failed; see {LOGS}/pip_install.log")
import importlib, ocsd; importlib.reload(ocsd)
from ocsd import hfcache
hfcache.repair(os.environ["HF_HOME"])   # delete broken model files so they download again
from ocsd import runlog
runlog.start(os.path.join(DRIVE_ROOT, "results"), CODE)   # every print below also goes to results/logs/run_*.log
from ocsd.config import Paths, ExperimentConfig, TIERS
P = Paths(DRIVE_ROOT); P.makedirs()
E = ExperimentConfig(backbone=BACKBONE, tier=TIER)
E.save(os.path.join(P.results, "experiment_config.json"))
print(E)

In [ ]:
#@title Khởi chạy ứng dụng
!pip install -q gradio
from ocsd.app import launch
launch(backbone=E.backbone, share=True)